In [9]:
import numpy as np
import pandas as pd
from markovstates.models import WeatherModel, HMMWeatherModel
from markovstates.utils import FeatMat, FINAL_FEATURES, Preprocess, hourly_dataframe

FM = FeatMat(hourly_dataframe, FINAL_FEATURES)
X = FM.construct_feat_mat()
print(FM.df.index[:5])
print(X.shape)
print(X[:5])
print(X.mean(axis=0))
print(X.std(axis=0))

RangeIndex(start=0, stop=5, step=1)
(1097, 3)
[[ 0.48172131  0.73038161 -1.92739153]
 [ 0.33749396  0.46799135 -1.1038959 ]
 [-0.05376393 -0.41122282 -0.56801075]
 [-0.48493072  0.03453048 -2.31666183]
 [-0.15454634  0.03638106 -0.74067914]]
[4.76355685e-08 1.45067796e-06 1.07172667e-08]
[1.00000004 1.         1.00000002]


In [10]:
hmm = HMMWeatherModel(n_components=5,covar_type='diag',n_restarts=20)

hmm.fit(X)
print(hmm.score(X))
print(hmm.predict(X))

-2872.7457549060564
[2 2 2 ... 2 2 3]


In [11]:
model_scores = hmm.score_table(X, n_range=(2,6))

print(model_scores)

  n_components log_likelihood        AIC        BIC
0            2     -3705.1189  7440.2377  7515.2427
1            3     -3292.0109  6636.0219  6766.0306
2            4     -3704.4479  7486.8958  7681.9088
3            5       -2897.69    5903.38  6173.3981


In [12]:
hmm = HMMWeatherModel(n_components=5, covar_type='full', n_restarts=50)

hmm.fit(X)
print(hmm.score(X))
print(hmm.predict(X))
print(hmm.score_table(X, n_range=(2,6)))

-2653.2069204909967
[2 2 2 ... 2 2 2]
  n_components log_likelihood        AIC        BIC
0            2     -3427.0101  6896.0202  7001.0272
1            3      -3011.332   6092.664  6267.6757
2            4      -2813.316  5728.6319   5983.649
3            5     -2846.1665   5830.333  6175.3561


In [13]:
unique, counts = np.unique(hmm.predict(X), return_counts=True)
print(dict(zip(unique, counts)))
print(X.shape)

{np.int64(0): np.int64(161), np.int64(1): np.int64(227), np.int64(2): np.int64(161), np.int64(3): np.int64(291), np.int64(4): np.int64(257)}
(1097, 3)


In [ ]:
pp = Preprocess(hourly_dataframe)
daily = pp.resample()
daily["regime"] = hmm.predict(X)
print(daily.groupby("regime")[FINAL_FEATURES].mean())
hmm.save("/Users/philipalexopoulos/markovstates/models/hmm_final.pkl")


        temperature_2m  surface_pressure  dew_point_2m
regime                                                
0            18.407957        940.771362     10.891499
1            11.083233        941.329224      6.225560
2            18.285818        941.668091      3.193621
3             6.901804        940.638611      3.840972
4            27.720695        942.228333      7.996400
